In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from edc_pdutils.dataframes import get_crf
from intecomm_analytics.dataframes import get_all_complications_df, get_appt_df, \
    get_medications_df
from intecomm_analytics.dataframes.main_1858_to_stata import df_main_variable_labels
from edc_analytics.stata import get_stata_labels_from_model

In [ ]:
df_appt = get_appt_df()

In [ ]:
variable_labels = {}
system_columns = ["id", "consent_model", "consent_version", "crf_status", "crf_status_comments", "created", "modified", "user_created", "user_modified", "hostname_created", "hostname_modified", "device_created", "device_modified", "locale_created", "locale_modified", "revision"]
df_appt_columns = [col for col in df_appt.columns.tolist() if col not in ["subject_visit_id", "appointment_id"]]

In [ ]:
df = df_appt.copy().reset_index(drop=True)

In [ ]:
for model in [
    "eq5d3l",
    "icecapa",
    ("healtheconomicsassets", "hhassets"),
    ("healtheconomicshouseholdhead", "hhhead"),
    ("healtheconomicsincome", "hhincome"),
    ("healtheconomicspatient", "patient"),
    ("healtheconomicsproperty", "hhproperty"),
    ("careseekinga", "csa"),
    ("careseekingb", "csb"),
    ("subjectvisitmissed", "mv"),
]:
    orig_columns = df.columns.tolist()
    try:
        model, suffix = model
    except ValueError:
        model, suffix = model, model
    df_crf = get_crf(f"intecomm_subject.{model}", subject_visit_model="intecomm_subject.subjectvisit", read_verbose=False, drop_action_item_columns=True)
    df_crf = (df_crf
        # df_crf[[col for col in df_crf.columns if col not in df_appt_columns and col not in system_columns]]
        # .copy()
        .drop(columns=["visit_datetime", "site_id", "visit_code_str", "reason", "reason_unscheduled", "reason_unscheduled_other", "reason_missed", "reason_missed_other", "endline_visit_code", "endline_visit_code_str", "endline_visit_datetime", "followup_days"])
        .drop(columns=system_columns)
        .rename(columns={col:f"{col}_{suffix}" for col in df_crf.columns if col not in df_appt_columns and col not in ["subject_visit_id", "appointment_id"]})
    )
    df_crf[f"crf_{model}"] = 1
    df_crf = df_crf.fillna(pd.NA)
    df = df.merge(df_crf, on="appointment_id", how="left", suffixes=("", "_y"))
    df = df.drop(columns = [col for col in df.columns if col.endswith("_y")])

    assert len(df) == 20953
    variable_labels.update(**get_stata_labels_from_model(df_appt, f"intecomm_subject.{model}", suffix))


In [ ]:
df_complications, df_complications_variable_labels = get_all_complications_df()
df = df.merge(df_complications, on="subject_visit_id", how="left", suffixes=("", "_y"))
df = df.drop(columns=[col for col in df.columns if col.endswith("_y")])
assert len(df) == 20953
variable_labels.update(**df_complications_variable_labels)


In [ ]:
df_meds, medication_variable_labels = get_medications_df()
df = df.merge(df_meds, on="subject_visit_id", how="left", suffixes=("", "_y"))
df = df.drop(columns=[col for col in df.columns if col.endswith("_y")])
assert len(df) == 20953
variable_labels.update(**medication_variable_labels)


In [ ]:
df_vitals = get_crf(model="intecomm_subject.vitals",subject_visit_model="intecomm_subject.subjectvisit", read_verbose=False)
df_vitals = df_vitals.sort_values(["subject_identifier", "visit_code"])
df_vitals[["weight", "height"]] = df_vitals.groupby("subject_identifier")[["weight", "height"]].ffill()
df_vitals["bmi"] = (df_vitals["weight"]) / ((df_vitals["height"] / 100) ** 2)
df = df.merge(df_vitals[["subject_visit_id", "weight","temperature", "height", "bmi", "sys_blood_pressure_avg", "dia_blood_pressure_avg", "severe_htn"]], on="subject_visit_id", how="left", suffixes=("_x", ""))
df = df.drop(columns=[col for col in df.columns if col.endswith("_x")])
variable_labels.update(**get_stata_labels_from_model(df_appt, f"intecomm_subject.vitals", None))

In [ ]:
# these are the long field names
# go through this by hand and shorten
original_labels = [
    'primary_vl_controlled_baseline_400',
    'primary_vl_controlled_baseline_50',
    'primary_vl_controlled_endline_50',
    'primary_vl_controlled_endline_400',
    'health_today_score_confirmed_eq5d3l',
    'external_wall_material_other_hhassets',
    'external_window_material_hhassets',
    'external_window_material_other_hhassets',
    'med_not_collected_reason_other_csa',
    'med_not_collected_reason_other_csb',
    'inpatient_household_nowork_days_csb',
    'inpatient_money_sources_other_csb',
    'inpatient_money_sources_main_other_csb',
    'rental_income_value_known_hhincome',
    'ngo_assistance_value_known_hhincome',
    'internal_remit_value_known_hhincome',
    'external_remit_value_known_hhincome',
    'more_sources_value_known_hhincome',
    'external_remit_currency_other_hhincome',
    'financial_status_compare_hhincome',
    'pat_employment_type_other_patient',
    'land_surface_area_units_hhproperty',
    'calculated_land_surface_area_hhproperty'
]
shortened_labels = [
    'primary_vl_cntrl_baseline_400',
    'primary_vl_cntrl_baseline_50',
    'primary_vl_cntrl_endline_50',
    'primary_vl_cntrl_endline_400',
    'health_today_score_conf_eq5d3l',
    'ext_wall_materl_other_hhassets',
    'ext_window_materl_hhassets',
    'ext_window_materl_other_hhassets',
    'med_not_collect_reason_other_csa',
    'med_not_collect_reason_other_csb',
    'inpatient_hh_nowork_days_csb',
    'inpatient_mny_src_other_csb',
    'inpatient_mny_src_main_other_csb',
    'rental_income_val_known_hhincome',
    'ngo_asst_val_known_hhincome',
    'int_remit_val_known_hhincome',
    'ext_remit_val_known_hhincome',
    'more_src_val_known_hhincome',
    'ext_remit_curr_other_hhincome',
    'financl_status_compare_hhincome',
    'pat_emply_type_other_patient',
    'land_surf_area_units_hhproperty',
    'calc_land_surf_area_hhproperty'
]

In [ ]:
# export
df["roof_material_other_hhassets"] = df["roof_material_other_hhassets"].fillna("")
df["tests_not_done_other_csa"] = df["tests_not_done_other_csa"].fillna("")
df["no_accessed_care_other_csb"] = df["no_accessed_care_other_csb"].fillna("")
df["hoh_education_other_hhhead"] = df["hoh_education_other_hhhead"].fillna("")
df["land_surface_area_hhproperty"] = df["land_surface_area_hhproperty"].astype("Float64")
df["calculated_land_surface_area_hhproperty"] = df["calculated_land_surface_area_hhproperty"].astype("Float64")

# rename cols in the Dataframe using shortened col names
rename_cols = dict(zip(original_labels, shortened_labels))
df = df.rename(columns=rename_cols)

In [ ]:
# update the variable labels for stata with the shortened col names
rev_variable_labels = {description: fld for fld, description in variable_labels.items()}
for orig_fld, shortened_fld in rename_cols.items():
    if orig_fld in variable_labels:
        rev_variable_labels[variable_labels[orig_fld]] = shortened_fld
variable_labels = {v:k for k,v in rev_variable_labels.items()}

In [ ]:
variable_labels.update(**df_main_variable_labels())

In [ ]:
for col in df.select_dtypes(include="timedelta").columns:
    df[col] = df[col].dt.total_seconds()

In [ ]:
df = df.astype({col: "Float64" for col in df.select_dtypes(include=["float", "float64"]).columns})
df = df.astype({col: "Int64" for col in df.select_dtypes(include=["int", "int64"]).columns})
df = df.astype({col: "datetime64[ns]" for col in df.select_dtypes(include=["datetime", "datetime64"]).columns})
df = df.fillna(pd.NA)

In [ ]:
df.select_dtypes(include="object").columns.tolist()

In [ ]:
df = df.astype({col: str for col in df.select_dtypes(include="object").columns})
# maybe ???
df = df.replace("nan", pd.NA)

In [ ]:
df = df.drop(columns=[
    "action_identifier_mv",
    "action_item_mv",
    "action_item_reason_mv",
    "appt_close_datetime",
    "appt_date_mv",
    "appt_type_other",
    "document_status",
    "document_status_comments",
    # "index",
    "is_confirmed",
    "parent_action_item_mv",
    "parent_action_identifier_mv",
    "related_action_item_mv",
    "related_action_identifier_mv",
])

In [ ]:
from datetime import datetime
timestamp =  datetime.now().strftime("%Y%m%d%H%M")
df.to_stata(
    path=analysis_folder / f"df_he_{timestamp}.dta",
    variable_labels=variable_labels,
    version=118,
    write_index=False,
)